# 나라 맞추기 게임 (V3)

## 게임 설명
1. **테마 기반 출제**: '유럽 국가', '아시아의 섬나라' 등 테마(`theme`)와 출제 문제 수(`quiz_count`)를 입력하면 대상 국가를 선정합니다.
2. **미스터리 힌트 (Tool 연동)**: 위키피디아 검색을 통해 각 국가의 정보를 수집하고, 정답(국가명)을 블라인드 처리한 미스터리 힌트 퀴즈를 병렬(`Send` API)로 생성합니다.
3. **대화형 퀴즈**: 사용자에게 힌트를 제공하고 정답 제출을 대기(`interrupt`)합니다.
4. **오답 재도전 (Conditional Edge)**: 만점(또는 특정 점수)에 도달하지 못하면 틀린 문제에 한해 '첫 글자 힌트' 등 추가 힌트를 제공하며 재도전 루프를 돕습니다.


In [ ]:
# 필요한 패키지 (사전 설치 확인용)
# %pip install wikipedia langchain-community pydantic langgraph langchain python-dotenv


In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 LANGCHAIN_API_KEY 및 OPENAI_API_KEY 등 로드
load_dotenv()

# LangSmith 연동 설정
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Country_Guessing_Game"  # 랭스미스에 표시될 프로젝트 이름


In [ ]:
import operator
from typing import TypedDict, List, Dict, Any
from typing_extensions import Annotated
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Send, Command
from langgraph.checkpoint.memory import InMemorySaver

from langchain.chat_models import init_chat_model
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# LLM 및 위키피디아 툴 셋업
llm = init_chat_model("openai:gpt-4o-mini")
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="ko", top_k_results=1, doc_content_chars_max=1000))
llm_with_tools = llm.bind_tools([wiki_tool])


In [ ]:
# 1. State 및 스키마 정의
class State(TypedDict):
    theme: str                     # 예: "유럽 국가"
    quiz_count: int                # 출제할 문제 수 (기본 3)
    target_countries: List[str]    # 대상 국가 이름 (정답 목록)
    quizzes: Annotated[List[Dict[str, str]], operator.add]  # 병렬 취합용 퀴즈 목록 (question, answer)
    user_answers: List[str]        # 사용자가 제출한 정답 목록
    evaluation_report: str         # 채점 결과 보고서
    score: int                     # 최종 점수
    hint: str                      # 오답 재도전 힌트 메시지

# 병렬 처리를 위한 서브 태스크 상태
class CountryTask(TypedDict):
    country: str

# 구조화된 출력을 위한 퀴즈 포맷
class CountryQuiz(BaseModel):
    question: str = Field(description="나라 이름을 직접 언급하지 않고, 랜드마크, 수도, 특징 등을 설명하여 나라를 맞추게 하는 퀴즈 문장")
    answer: str = Field(description="실제 정답 (나라 이름)")


In [ ]:
# 2. 게임 노드 및 엣지 로직 구현

def select_countries(state: State) -> Dict[str, Any]:
    print("--- [노드] select_countries: 테마 기반 국가 선정 ---")
    theme = state.get("theme", "전세계 유명 국가")
    count = state.get("quiz_count", 3)
    
    prompt = f"""
    주제: '{theme}'
    위 주제에 부합하는 나라 이름을 정확히 {count}개 추천해주세요.
    부연 설명이나 기호 없이 한 줄에 한 국가 이름만 작성해주세요.
    """
    response = llm.invoke(prompt)
    countries = [line.strip() for line in response.content.split("\n") if line.strip()]
    countries = countries[:count]  # 요청 개수 슬라이싱
    return {"target_countries": countries}

def dispatch_quizzes(state: State):
    countries = state.get("target_countries", [])
    print(f"--- [라우팅] {len(countries)}개 국가 퀴즈 생성을 병렬 배분 ---")
    return [Send("generate_country_quiz", {"country": c}) for c in countries]

def generate_country_quiz(task: CountryTask) -> Dict[str, Any]:
    target = task["country"]
    print(f"--- [노드(병렬)] generate_country_quiz: '{target}' 퀴즈 생성 중 ---")
    
    # 위키피디아 검색으로 국가 정보 확보
    try:
        wiki_info = wiki_tool.invoke({"query": target})
    except Exception:
        wiki_info = "해당 국가에 대한 상세 정보 검색 실패."

    prompt = f"""
    위키피디아 정보:
    {wiki_info}
    
    이 정보를 바탕으로, 해당 국가가 어디인지 맞출 수 있는 퀴즈를 생성하세요.
    절대 질문 안에 정답 국가 이름({target})이 직접적으로 노출되어서는 안 됩니다.
    """
    
    structured_llm = llm.with_structured_output(CountryQuiz)
    quiz_data = structured_llm.invoke(prompt)
    
    return {
        "quizzes": [{
            "question": quiz_data.question,
            "answer": target  # 실제 타겟을 정답으로 명시 보장
        }]
    }

def aggregate_quizzes(state: State) -> Dict[str, Any]:
    # 병렬 처리가 끝난 후 동기화(fan-in) 역할을 하는 더미 노드
    print("--- [노드] aggregate_quizzes: 모든 병렬 생성 취합 완료 ---")
    return {}

def quiz_interrupt(state: State) -> Dict[str, Any]:
    print("--- [노드] quiz_interrupt (대기) ---")
    hint = state.get("hint", "")
    if hint:
        print(f"[추가 힌트 시스템]:\n{hint}")
        
    quizzes = state.get("quizzes", [])
    answer = interrupt({
        "questions": [c["question"] for c in quizzes],
        "instruction": "각 퀴즈 번호에 맞는 정답(나라 이름)을 리스트로 작성하여 'user_answers'에 담아 재개(resume)하세요."
    })
    return {"user_answers": answer.get("user_answers", [])}

def grade_quiz(state: State) -> Dict[str, Any]:
    print("--- [노드] grade_quiz: 채점 진행 ---")
    quizzes = state.get("quizzes", [])
    user_answers = state.get("user_answers", [])
    
    report_lines = ["### 게임 채점 결과"]
    correct_count = 0
    total = len(quizzes)
    
    for idx, (quiz, u_ans) in enumerate(zip(quizzes, user_answers)):
        correct_ans = str(quiz["answer"]).strip().lower()
        u_ans_clean = str(u_ans).strip().lower()
        
        # 간단한 포함(in) 확인 로직 (예: 정답 '프랑스', 유저가 '프랑스입니다' 입력 시 정답 처리)
        is_correct = correct_ans in u_ans_clean or u_ans_clean in correct_ans
        
        if is_correct:
            correct_count += 1
            report_lines.append(f"Q{idx+1}. 정답! ({correct_ans})")
        else:
            report_lines.append(f"Q{idx+1}. 오답 (제출: {u_ans_clean} / 정답: {correct_ans})")
            
    score = int((correct_count / total) * 100) if total > 0 else 0
    report_lines.append(f"\n최종 점수: {score}점")
    return {"evaluation_report": "\n".join(report_lines), "score": score}

def check_score(state: State) -> str:
    score = state.get("score", 0)
    print(f"--- [조건] check_score: 현재 점수 {score}점 ---")
    # 만점 시에만 종료
    if score >= 100:
        return "pass"
    return "fail"

def generate_extra_hint(state: State) -> Dict[str, Any]:
    print("--- [노드] generate_extra_hint: 오답 힌트 생성 ---")
    quizzes = state.get("quizzes", [])
    user_answers = state.get("user_answers", [])
    
    hint_lines = ["틀린 문제에 대한 힌트입니다."]
    for idx, (quiz, u_ans) in enumerate(zip(quizzes, user_answers)):
        correct_ans = str(quiz["answer"]).strip()
        u_ans_clean = str(u_ans).strip().lower()
        
        if correct_ans.lower() not in u_ans_clean:
            first_char = correct_ans[0] if len(correct_ans) > 0 else "?"
            hint_lines.append(f"Q{idx+1} 힌트: 정답은 '{first_char}' (으)로 시작합니다.")
            
    return {"hint": "\n".join(hint_lines)}


In [ ]:
# 3. 그래프 조립 및 컴파일
graph_builder = StateGraph(State)

graph_builder.add_node("select_countries", select_countries)
graph_builder.add_node("generate_country_quiz", generate_country_quiz)
graph_builder.add_node("aggregate_quizzes", aggregate_quizzes)
graph_builder.add_node("quiz_interrupt", quiz_interrupt)
graph_builder.add_node("grade_quiz", grade_quiz)
graph_builder.add_node("generate_extra_hint", generate_extra_hint)

graph_builder.add_edge(START, "select_countries")
graph_builder.add_conditional_edges("select_countries", dispatch_quizzes, ["generate_country_quiz"])
graph_builder.add_edge("generate_country_quiz", "aggregate_quizzes")
graph_builder.add_edge("aggregate_quizzes", "quiz_interrupt")
graph_builder.add_edge("quiz_interrupt", "grade_quiz")

graph_builder.add_conditional_edges("grade_quiz", check_score, {"pass": END, "fail": "generate_extra_hint"})
graph_builder.add_edge("generate_extra_hint", "quiz_interrupt")

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)


---
## 격리된 단위 테스트 셀 (노드별 단독 검증)


In [ ]:
# [단위 테스트 1] select_countries 테스트
test_state = {"theme": "스칸디나비아 반도 국가", "quiz_count": 2}
res1 = select_countries(test_state)
print("추출된 나라 목록:", res1)
assert "target_countries" in res1, "목록 누락"
assert len(res1["target_countries"]) <= 2


In [ ]:
# [단위 테스트 2] generate_country_quiz 테스트 (Tool 동작 검증)
if 'res1' in globals() and res1.get("target_countries"):
    target = res1["target_countries"][0]
else:
    target = "노르웨이"

test_task = {"country": target}
print(f"타겟 국가: {target}")
res2 = generate_country_quiz(test_task)
print("\n생성된 퀴즈 정보:")
print(res2["quizzes"][0])
assert "quizzes" in res2


In [ ]:
# [단위 테스트 3] grade_quiz 및 추가 힌트 생성 테스트
mock_quizzes = [
    {"question": "이 나라는 피오르드 지형이 유명합니다.", "answer": "노르웨이"},
    {"question": "이 나라는 파리가 수도입니다.", "answer": "프랑스"}
]
# 오답 주입 테스트 (노르웨이를 스웨덴으로 오답 처리)
test_state3 = {
    "quizzes": mock_quizzes,
    "user_answers": ["스웨덴", "프랑스입니다"]
}
res3 = grade_quiz(test_state3)
print("채점 리포트:\n", res3["evaluation_report"])

cond = check_score({"score": res3["score"]})
print("\n점수 판별 결과:", cond)

if cond == "fail":
    test_state3["user_answers"] = test_state3["user_answers"]  # user_answers 유지 확인
    res_hint = generate_extra_hint(test_state3)
    print("\n생성된 오답 힌트:\n", res_hint["hint"])


---
## 통합 테스트 (게임 시나리오 플레이)


In [ ]:
# [통합 테스트 1] 게임 시작 및 퀴즈 출제
config = {"configurable": {"thread_id": "country_game_session"}}
initial_state = {"theme": "유럽의 유명 관광 국가", "quiz_count": 2}

print("=== 나라 맞추기 게임 시작 ===")
result = graph.invoke(initial_state, config=config)
print("\n인터럽트 진입: 사용자 답변을 기다립니다.")


In [ ]:
# [통합 테스트 2] 답변 제출 및 결과 확인
# 주의: 위 셀 실행 후 생성된 퀴즈를 확인하고 답변을 주입하세요. (현재는 가상의 오답과 정답 제출 예시)

mock_answers = ["이탈리아", "스위스"] # 상황에 따라 점수가 갈림

final_result = graph.invoke(
    Command(resume={"user_answers": mock_answers}),
    config=config
)

print("\n=== 재개 후 게임 상태 ===")
print(final_result.get("evaluation_report"))

if final_result.get("hint"):
    print("\n[재도전 추가 힌트 발송!]")
    print(final_result.get("hint"))
